In [ ]:
import pandas as pd
import numpy as np

# 1. Carrega o arquivo pulando as linhas de cabeçalho do INEP
df_raw = pd.read_csv('Rendimendo_Escola_2024.csv', skiprows=8, low_memory=False)

# 2. Organiza os nomes das colunas de identificação
df_raw.columns = [
    'ano', 'regiao', 'uf', 'codigo_municipio', 'nome_municipio',
    'codigo_escola', 'nome_escola', 'localizacao', 'dependencia_adm'
] + list(df_raw.columns[9:])

# 3. Mapeia as colunas de Reprovação (2_CAT) e Abandono/Evasão (3_CAT)
colunas_identificacao = ['uf', 'codigo_escola', 'nome_escola', 'localizacao', 'dependencia_adm']
colunas_metricas = [c for c in df_raw.columns if str(c).startswith('2_') or str(c).startswith('3_')]
df_combinado = df_raw[colunas_identificacao + colunas_metricas].copy()

# 4. Renomeia os TOTAIS de cada bloco
df_combinado = df_combinado.rename(columns={
    '2_CAT_FUN': 'reprovacao_fundamental_total',
    '2_CAT_MED': 'reprovacao_medio_total',
    '3_CAT_FUN': 'evasao_fundamental_total',
    '3_CAT_MED': 'evasao_medio_total'
})

# 5. Limpa a sujeira ('--') e converte as taxas para números reais (float)
colunas_para_converter = [
    'reprovacao_fundamental_total', 'reprovacao_medio_total',
    'evasao_fundamental_total', 'evasao_medio_total'
]
for col in colunas_para_converter:
    df_combinado[col] = pd.to_numeric(df_combinado[col].replace('--', np.nan), errors='coerce')

# Limpa o ID da escola para inteiro
df_combinado = df_combinado.dropna(subset=['codigo_escola'])
df_combinado['codigo_escola'] = df_combinado['codigo_escola'].astype(int)

# 6. JUNTA AS DEPENDÊNCIAS EM "PÚBLICA"
mapeamento_publico = {
    'Federal': 'Pública',
    'Estadual': 'Pública',
    'Municipal': 'Pública',
    'Privada': 'Privada'
}
df_combinado['dependencia_adm'] = df_combinado['dependencia_adm'].replace(mapeamento_publico)

# Isola a tabela final exatamente com o que a equipe quer
tabela_equipe = df_combinado[[
    'uf', 'codigo_escola', 'nome_escola','dependencia_adm',
    'reprovacao_fundamental_total', 'reprovacao_medio_total',
    'evasao_fundamental_total', 'evasao_medio_total'
]].reset_index(drop=True)

# ==============================================================================
# LINHA DE EXPORTAÇÃO (A MÁGICA)
# ==============================================================================
# index=False evita que o Pandas crie uma coluna de números inúteis na esquerda
# sep=';' ajuda o Excel brasileiro a separar as colunas automaticamente
tabela_equipe.to_csv('tabela_evasao_reprovacao_publica_2024.csv', index=False, sep=';', encoding='utf-8-sig')

print("Tabela exportada com sucesso! Procure pelo arquivo na barra lateral esquerda do Colab.")

Tabela exportada com sucesso! Procure pelo arquivo na barra lateral esquerda do Colab.
